In [2]:
library(xgboost)
library(Matrix)
library(ggplot2)
library(lattice)
library(caret)
library(dplyr)
library(tictoc)

In [1]:
train_values <- read.csv("train_values.csv",stringsAsFactors = T)
test_values <- read.csv("test_values.csv",stringsAsFactors = T)
train_labels <- read.csv("train_labels.csv",stringsAsFactors = T)
submission_format <- read.csv("submission_format.csv",stringsAsFactors = T)

In [4]:
data <- merge(train_values,train_labels,by=c('building_id','building_id'),all.x=T)
test <- test_values

In [5]:
geo_level_1_damage <- 0:30
geo_level_2_damage <- 0:1427
geo_level_3_damage <- 0:12567

agg_geo_level_1 <- data %>% group_by(geo_level_1_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

agg_geo_level_2 <- data %>% group_by(geo_level_2_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

agg_geo_level_3 <- data %>% group_by(geo_level_3_id) %>% 
  summarise(mean_damage=mean(damage_grade),
            .groups = 'drop')

for (i in 1:31){geo_level_1_damage[i] <- agg_geo_level_1[i,2]}  

k <- 1
for (i in 1:1428){if(i-1 == agg_geo_level_2[k,1]){ 
        geo_level_2_damage[i] <- agg_geo_level_2[k,2]
        k <- k+1 } else { geo_level_2_damage[i] <- NA}}

k <- 1
for (i in 1:12568){if(i-1 == agg_geo_level_3[k,1]){ 
        geo_level_3_damage[i] <- agg_geo_level_3[k,2]
        k <- k+1 } else { geo_level_3_damage[i] <- NA}}

for (i in 1:260601){
    if(is.na(geo_level_3_damage[data[i,c("geo_level_3_id")]+1])){
        if(is.na(geo_level_2_damage[data[i,c("geo_level_2_id")]+1])){
            data$geo_level_damage[i] <- as.numeric(unlist(geo_level_1_damage[data[i,c("geo_level_1_id")]+1]))
        } else {data$geo_level_damage[i] <- as.numeric(unlist(geo_level_2_damage[data[i,c("geo_level_2_id")]+1]))}
    } else {data$geo_level_damage[i] <- as.numeric(unlist(geo_level_3_damage[data[i,c("geo_level_3_id")]+1]))}
}

for (i in 1:86868){
    if(is.na(geo_level_3_damage[test[i,c("geo_level_3_id")]+1])){
        if(is.na(geo_level_2_damage[test[i,c("geo_level_2_id")]+1])){
            test$geo_level_damage[i] <- as.numeric(unlist(geo_level_1_damage[test[i,c("geo_level_1_id")]+1]))
        } else {test$geo_level_damage[i] <- as.numeric(unlist(geo_level_2_damage[test[i,c("geo_level_2_id")]+1]))}
    } else {test$geo_level_damage[i] <- as.numeric(unlist(geo_level_3_damage[test[i,c("geo_level_3_id")]+1]))}
}

In [6]:
data <- data[,-c(1,2,3,4)]
test <- test[,-c(1,2,3,4)]
nzv <- nearZeroVar(data)
data <- data[, -nzv]
test <- test[, -nzv]

In [7]:
X <- sparse.model.matrix(damage_grade ~ .,data = data)[,-1]
y <- as.numeric(data[,c("damage_grade")]-1)

In [104]:
true.test <- sparse.model.matrix( ~ .,data = test)[,-1]

In [106]:
dim(X)

[1] 260601     32

In [105]:
dim(true.test)

[1] 86868    32

In [102]:
?sparse.model.matrix

On split la data en 80/20 pour evaluer l'efficasiter de notre model 

In [16]:
size <- nrow(X)
spliting_idx <- sample(1:size)
split <- floor(size*0.8)

X.train <- X[spliting_idx[1:split],]
X.test <- X[spliting_idx[(split+1):size],]
y.train <- y[spliting_idx[1:split]]
y.test <- y[spliting_idx[(split+1):size]]

On teste le model par défaut 

on fait pas de cross validation et on fait pas de teste intenrne pour entrainer le model sur tout la data (X.train,y.train)

In [35]:
tic()
grid_default <- expand.grid(
  nrounds = 100,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE # FALSE for reproducible results 
    #summaryFunction = multiClassSummary
)

xgb_base <- caret::train(
    x = X.train,
    y = y.train,
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    #objective = "multi:softmax", #3 fois plus slow quand on met cesi et a pas l'air de changer grand chose au resulta 
    #num_class=3
)
toc()

21.324 sec elapsed


In [26]:
pred_base <- predict(xgb_base, X.test)

In [27]:
print(head(pred))

[1] 1.0068665 1.3476576 1.4471692 1.0762129 1.4764630 0.6184264


In [ ]:
basePred <- predict(xgb_base,testMNX)
baseRMSE <- caret::RMSE(basePred, testMNY)
baseRMSE

On a l'air d'avoir un truc qui marche pour le multiclass ici 

In [93]:
tic()
grid_default <- expand.grid(
  nrounds = 100,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    #"eval_metric" = "mlogloss"
)
toc()

60.98 sec elapsed


In [59]:
pred_base <- predict(xgb_base, X.test)
pred_base[1:100]

[1] 1 1 1 1 2 1 2 1 2 1 1 1 2 1 2 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 2 2 1 2 2 1
 [38] 1 1 2 1 0 2 1 1 1 2 2 1 1 1 1 1 2 1 2 1 1 1 2 2 1 1 2 1 2 1 1 1 1 1 1 1 1
 [75] 1 1 0 1 2 1 1 1 1 1 2 1 2 2 1 0 1 1 1 2 0 2 2 1 1 1
Levels: 0 1 2

In [60]:
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],y.test[j]] <- cfm[pred_base[j],y.test[j]]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.1512437


Le F1 scorre est vraiment pas terrible 

In [65]:
# note to start nrounds from 200, as smaller learning rates result in errors so
# big with lower starting points that they'll mess the scales
tune_grid <- expand.grid(
  nrounds = seq(200, 1000, 200),
  eta = 0.3,
  max_depth = 6,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

tune_control <- caret::trainControl(
  method = "none", #"cv", # cross-validation
  #number = 3, # with n folds 
  #index = createFolds(tr_treated$Id_clean), # fix the folds
  verboseIter = FALSE, # no training log
  allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_tune <- caret::train(
    x = X.train,
    y = as.factor(y.train),
  trControl = tune_control,
  tuneGrid = tune_grid,
  method = "xgbTree",
  verbose = TRUE,
    nthread = 4
)

# helper function for the plots
tuneplot <- function(x, probs = .90) {
  ggplot(x) +
    coord_cartesian(ylim = c(quantile(x$results$RMSE, probs = probs), min(x$results$RMSE))) +
    theme_bw()
}

tuneplot(xgb_tune)

ERROR: Error: Only one model should be specified in tuneGrid with no resampling


In [71]:
tic()
grid_default <- expand.grid(
  nrounds = c(200,400),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 3,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

[13:05:01] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[13:07:40] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[13:10:19] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
597.824 sec elapsed


In [112]:
?caret::trainControl

In [94]:
pred_base <- predict(xgb_base, X.test)
#pred_base <- as.numeric(pred_base)

In [96]:
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7520577


In [97]:
pred_base

[1] 1 1 1 1 2 1 2 1 2 1 1 1 2 1 2 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 2 2 1 2 2
   [37] 1 1 1 2 1 0 2 1 1 1 2 2 1 1 1 1 1 2 1 2 1 1 1 2 2 1 1 2 1 2 1 1 1 1 1 1
   [73] 1 1 1 1 0 1 2 1 1 1 1 1 2 1 2 2 1 0 1 1 1 2 0 2 2 1 1 1 1 1 1 1 1 1 1 2
  [109] 1 1 1 0 2 2 1 1 1 1 1 0 2 2 1 1 1 2 2 1 2 1 0 1 1 1 1 1 1 2 1 1 1 0 1 1
  [145] 1 1 1 2 1 2 1 1 0 1 1 1 2 1 2 1 2 1 0 2 2 2 1 1 1 1 1 1 1 1 2 1 1 0 1 2
  [181] 2 1 0 1 1 2 1 0 2 1 1 1 0 2 1 1 2 2 2 1 1 1 1 0 1 2 0 0 1 2 1 1 1 1 1 0
  [217] 2 1 1 2 1 1 2 1 1 1 1 1 2 0 1 2 1 1 2 0 1 1 1 1 2 1 0 1 1 2 2 1 1 2 1 2
  [253] 2 0 1 1 1 2 1 2 1 1 1 1 2 2 1 2 2 1 0 1 1 1 2 0 1 1 2 2 1 1 1 2 1 1 2 1
  [289] 1 1 0 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1 2 1 2 1 2 2 1 1 2 1 1 1 1 0 1 1 1 1
  [325] 1 1 1 1 2 1 1 2 1 1 1 1 1 0 1 1 1 0 2 1 2 0 2 1 1 1 1 2 2 1 1 2 1 1 2 2
  [361] 1 2 1 1 1 1 1 2 2 1 1 1 1 2 2 1 2 2 1 1 0 0 1 1 1 1 1 0 1 2 1 0 1 1 1 1
  [397] 1 1 1 1 1 1 2 2 2 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 2 2
  [433] 2 2 2 1 1 2 1 0 0 2 1 2 1 2 1 1 2 1 1 0 2 0 1 2 1 1 1 0 1 2 2 2 1 1 0 1
  [469] 1 2 2 1 2 2 1 1 1 1 1 1 1 2 0 1 1 1 0 2 2 2 1 1 1 2 1 0 2 1 1 1 1 1 1 1
  [505] 2 1 1 1 1 1 1 1 2 1 1 0 1 0 1 1 2 1 0 2 1 2 1 1 2 1 1 1 1 1 2 1 1 1 2 1
  [541] 1 1 1 1 1 1 1 1 2 1 2 1 1 2 1 1 1 1 1 0 1 1 2 1 1 2 1 1 1 1 2 0 0 2 1 1
  [577] 1 1 1 2 2 2 2 2 2 1 2 1 1 1 1 1 1 1 1 2 1 2 1 2 2 1 2 1 1 1 2 1 1 0 2 1
  [613] 2 1 1 1 1 1 2 0 1 1 2 2 1 1 2 1 1 2 1 1 1 1 2 1 2 1 1 2 1 0 2 1 1 2 1 1
  [649] 1 1 2 1 1 0 1 1 1 2 1 2 2 1 1 2 1 1 1 2 2 2 2 1 2 1 1 2 1 1 1 2 1 1 0 1
  [685] 1 1 2 1 1 2 1 1 2 1 1 2 1 2 1 0 1 1 2 1 1 1 1 1 2 1 1 1 2 1 1 1 1 1 2 1
  [721] 1 1 1 0 1 1 2 1 2 1 0 2 1 1 2 1 1 2 2 2 1 1 1 2 2 2 2 1 1 2 1 2 2 1 1 2
  [757] 1 2 1 2 1 1 2 1 2 2 1 2 1 2 2 1 1 0 1 2 1 1 1 1 1 1 2 2 2 1 1 1 0 1 1 2
  [793] 1 0 1 1 1 1 1 1 2 1 2 2 2 1 0 1 1 1 1 1 2 1 1 2 2 2 1 1 2 1 2 1 1 1 1 1
  [829] 1 2 2 2 0 1 2 2 1 0 1 1 2 1 1 0 1 2 1 1 2 1 1 2 2 1 1 2 2 1 2 2 2 2 1 1
  [865] 1 1 2 2 1 2 1 2 2 1 1 2 1 1 1 1 1 1 2 0 1 2 2 1 1 1 0 1 1 2 1 1 1 1 1 2
  [901] 1 1 2 2 1 2 1 2 2 0 1 2 1 1 1 1 1 1 1 0 1 1 2 2 1 0 1 1 2 1 1 2 1 1 1 2
  [937] 1 1 1 1 1 1 1 1 1 1 2 1 2 1 0 1 1 1 2 1 1 1 1 2 1 1 1 1 0 2 1 1 1 1 0 1
  [973] 1 2 0 1 0 2 1 1 1 1 1 2 1 1 1 1 1 1 0 1 1 1 1 1 1 1 2 2 1 1 2 1 2 1 1 2
 [1009] 1 1 2 2 1 1 1 2 2 2 1 1 1 1 1 1 1 2 1 2 1 1 2 1 1 1 1 1 1 1 1 2 1 2 1 0
 [1045] 0 1 0 1 1 1 2 0 2 1 1 1 2 2 1 2 1 1 2 1 1 1 1 1 1 1 2 1 1 2 1 1 0 1 2 1
 [1081] 1 1 1 0 2 2 2 2 2 1 1 2 2 1 1 1 1 1 0 1 1 1 1 1 0 1 1 2 1 1 2 2 2 1 1 2
 [1117] 1 0 1 1 0 2 1 1 1 1 2 0 1 1 2 1 1 1 1 1 1 1 1 1 2 1 1 0 2 1 1 1 2 2 1 1
 [1153] 1 0 1 1 1 2 2 0 1 2 1 1 1 1 2 2 1 1 1 2 1 1 0 2 1 1 1 1 1 1 2 1 1 0 1 1
 [1189] 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 1 1 1 1 1 1 2 2 2 1 1 1
 [1225] 0 1 1 1 1 1 1 2 1 1 1 1 2 2 1 2 2 1 1 1 2 1 2 1 2 2 2 2 1 1 1 2 1 1 2 1
 [1261] 2 1 2 1 2 1 1 2 1 1 1 1 2 1 0 1 1 1 1 2 1 2 2 1 1 0 0 1 2 1 2 1 2 2 1 1
 [1297] 1 2 1 2 1 1 1 2 1 1 2 2 0 1 1 2 2 1 1 0 1 1 1 1 2 1 2 1 1 2 1 1 1 1 2 1
 [1333] 1 1 1 1 2 2 1 1 1 1 0 1 1 2 1 1 1 1 2 1 2 0 2 2 1 1 1 1 1 1 1 0 2 2 2 1
 [1369] 1 1 2 1 1 1 1 1 2 1 2 1 1 2 1 2 1 2 0 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1
 [1405] 2 1 1 2 1 1 2 2 0 1 1 1 2 2 2 2 1 2 1 0 1 2 1 1 2 1 1 2 2 1 1 0 1 1 2 1
 [1441] 1 1 2 1 2 1 2 2 2 1 2 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 2 1 0 2 1 2 1 2 2 1
 [1477] 1 1 1 0 2 2 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 0 2 2 1 1 1 1 2
 [1513] 1 1 1 2 1 1 1 1 1 2 1 1 0 0 1 1 1 2 1 1 1 1 2 1 2 1 1 2 1 1 0 2 1 1 1 1
 [1549] 1 2 2 1 1 1 1 1 1 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 2 1 1 2 2 0 1 0
 [1585] 1 1 1 2 1 1 1 0 1 1 1 1 2 1 1 2 2 1 2 1 1 1 2 1 1 0 1 1 2 1 1 1 2 1 1 1
 [1621] 2 1 2 2 1 2 1 1 1 2 2 1 1 2 1 1 1 1 2 1 1 2 1 2 1 1 1 1 1 1 2 2 1 2 1 1
 [1657] 1 0 1 1 1 1 1 0 0 2 1 2 1 0 1 0 1 2 1 1 1 2 1 2 1 1 0 2 1 1 1 2 2 1 2 1
 [1693] 1 2 1 1 1 1 1 1 1 1 1 1 1 2 0 1 1 2 1 1 1 1 1 2 1 1 2 1 1 1 1 1 2 1 1 1
 [1729] 1 2 1 0 2 1 1 1 2 0 2 0 2 1 1 1 2 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2
 [1765] 1 2 2 0 2 1 2 0 1 1 2 1 1 2 1 1 2 1 1 1 1 1 1 1 1 0 1 2 0 1 0 1 1 1 2 1
 [18

In [98]:
as.numeric(pred_base)

[1] 2 2 2 2 3 2 3 2 3 2 2 2 3 2 3 2 2 1 2 2 2 2 1 2 2 2 2 2 2 2 2 3 3 2 3 3
   [37] 2 2 2 3 2 1 3 2 2 2 3 3 2 2 2 2 2 3 2 3 2 2 2 3 3 2 2 3 2 3 2 2 2 2 2 2
   [73] 2 2 2 2 1 2 3 2 2 2 2 2 3 2 3 3 2 1 2 2 2 3 1 3 3 2 2 2 2 2 2 2 2 2 2 3
  [109] 2 2 2 1 3 3 2 2 2 2 2 1 3 3 2 2 2 3 3 2 3 2 1 2 2 2 2 2 2 3 2 2 2 1 2 2
  [145] 2 2 2 3 2 3 2 2 1 2 2 2 3 2 3 2 3 2 1 3 3 3 2 2 2 2 2 2 2 2 3 2 2 1 2 3
  [181] 3 2 1 2 2 3 2 1 3 2 2 2 1 3 2 2 3 3 3 2 2 2 2 1 2 3 1 1 2 3 2 2 2 2 2 1
  [217] 3 2 2 3 2 2 3 2 2 2 2 2 3 1 2 3 2 2 3 1 2 2 2 2 3 2 1 2 2 3 3 2 2 3 2 3
  [253] 3 1 2 2 2 3 2 3 2 2 2 2 3 3 2 3 3 2 1 2 2 2 3 1 2 2 3 3 2 2 2 3 2 2 3 2
  [289] 2 2 1 1 2 2 2 1 2 2 2 2 2 2 2 2 2 2 3 2 3 2 3 3 2 2 3 2 2 2 2 1 2 2 2 2
  [325] 2 2 2 2 3 2 2 3 2 2 2 2 2 1 2 2 2 1 3 2 3 1 3 2 2 2 2 3 3 2 2 3 2 2 3 3
  [361] 2 3 2 2 2 2 2 3 3 2 2 2 2 3 3 2 3 3 2 2 1 1 2 2 2 2 2 1 2 3 2 1 2 2 2 2
  [397] 2 2 2 2 2 2 3 3 3 2 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 2 3 3
  [433] 3 3 3 2 2 3 2 1 1 3 2 3 2 3 2 2 3 2 2 1 3 1 2 3 2 2 2 1 2 3 3 3 2 2 1 2
  [469] 2 3 3 2 3 3 2 2 2 2 2 2 2 3 1 2 2 2 1 3 3 3 2 2 2 3 2 1 3 2 2 2 2 2 2 2
  [505] 3 2 2 2 2 2 2 2 3 2 2 1 2 1 2 2 3 2 1 3 2 3 2 2 3 2 2 2 2 2 3 2 2 2 3 2
  [541] 2 2 2 2 2 2 2 2 3 2 3 2 2 3 2 2 2 2 2 1 2 2 3 2 2 3 2 2 2 2 3 1 1 3 2 2
  [577] 2 2 2 3 3 3 3 3 3 2 3 2 2 2 2 2 2 2 2 3 2 3 2 3 3 2 3 2 2 2 3 2 2 1 3 2
  [613] 3 2 2 2 2 2 3 1 2 2 3 3 2 2 3 2 2 3 2 2 2 2 3 2 3 2 2 3 2 1 3 2 2 3 2 2
  [649] 2 2 3 2 2 1 2 2 2 3 2 3 3 2 2 3 2 2 2 3 3 3 3 2 3 2 2 3 2 2 2 3 2 2 1 2
  [685] 2 2 3 2 2 3 2 2 3 2 2 3 2 3 2 1 2 2 3 2 2 2 2 2 3 2 2 2 3 2 2 2 2 2 3 2
  [721] 2 2 2 1 2 2 3 2 3 2 1 3 2 2 3 2 2 3 3 3 2 2 2 3 3 3 3 2 2 3 2 3 3 2 2 3
  [757] 2 3 2 3 2 2 3 2 3 3 2 3 2 3 3 2 2 1 2 3 2 2 2 2 2 2 3 3 3 2 2 2 1 2 2 3
  [793] 2 1 2 2 2 2 2 2 3 2 3 3 3 2 1 2 2 2 2 2 3 2 2 3 3 3 2 2 3 2 3 2 2 2 2 2
  [829] 2 3 3 3 1 2 3 3 2 1 2 2 3 2 2 1 2 3 2 2 3 2 2 3 3 2 2 3 3 2 3 3 3 3 2 2
  [865] 2 2 3 3 2 3 2 3 3 2 2 3 2 2 2 2 2 2 3 1 2 3 3 2 2 2 1 2 2 3 2 2 2 2 2 3
  [901] 2 2 3 3 2 3 2 3 3 1 2 3 2 2 2 2 2 2 2 1 2 2 3 3 2 1 2 2 3 2 2 3 2 2 2 3
  [937] 2 2 2 2 2 2 2 2 2 2 3 2 3 2 1 2 2 2 3 2 2 2 2 3 2 2 2 2 1 3 2 2 2 2 1 2
  [973] 2 3 1 2 1 3 2 2 2 2 2 3 2 2 2 2 2 2 1 2 2 2 2 2 2 2 3 3 2 2 3 2 3 2 2 3
 [1009] 2 2 3 3 2 2 2 3 3 3 2 2 2 2 2 2 2 3 2 3 2 2 3 2 2 2 2 2 2 2 2 3 2 3 2 1
 [1045] 1 2 1 2 2 2 3 1 3 2 2 2 3 3 2 3 2 2 3 2 2 2 2 2 2 2 3 2 2 3 2 2 1 2 3 2
 [1081] 2 2 2 1 3 3 3 3 3 2 2 3 3 2 2 2 2 2 1 2 2 2 2 2 1 2 2 3 2 2 3 3 3 2 2 3
 [1117] 2 1 2 2 1 3 2 2 2 2 3 1 2 2 3 2 2 2 2 2 2 2 2 2 3 2 2 1 3 2 2 2 3 3 2 2
 [1153] 2 1 2 2 2 3 3 1 2 3 2 2 2 2 3 3 2 2 2 3 2 2 1 3 2 2 2 2 2 2 3 2 2 1 2 2
 [1189] 2 2 2 2 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 2 2 2 2 2 2 3 3 3 2 2 2
 [1225] 1 2 2 2 2 2 2 3 2 2 2 2 3 3 2 3 3 2 2 2 3 2 3 2 3 3 3 3 2 2 2 3 2 2 3 2
 [1261] 3 2 3 2 3 2 2 3 2 2 2 2 3 2 1 2 2 2 2 3 2 3 3 2 2 1 1 2 3 2 3 2 3 3 2 2
 [1297] 2 3 2 3 2 2 2 3 2 2 3 3 1 2 2 3 3 2 2 1 2 2 2 2 3 2 3 2 2 3 2 2 2 2 3 2
 [1333] 2 2 2 2 3 3 2 2 2 2 1 2 2 3 2 2 2 2 3 2 3 1 3 3 2 2 2 2 2 2 2 1 3 3 3 2
 [1369] 2 2 3 2 2 2 2 2 3 2 3 2 2 3 2 3 2 3 1 2 2 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2
 [1405] 3 2 2 3 2 2 3 3 1 2 2 2 3 3 3 3 2 3 2 1 2 3 2 2 3 2 2 3 3 2 2 1 2 2 3 2
 [1441] 2 2 3 2 3 2 3 3 3 2 3 2 2 2 2 2 2 2 2 2 2 2 3 2 2 2 3 2 1 3 2 3 2 3 3 2
 [1477] 2 2 2 1 3 3 2 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 1 3 3 2 2 2 2 3
 [1513] 2 2 2 3 2 2 2 2 2 3 2 2 1 1 2 2 2 3 2 2 2 2 3 2 3 2 2 3 2 2 1 3 2 2 2 2
 [1549] 2 3 3 2 2 2 2 2 2 3 3 3 2 2 2 2 2 2 2 2 2 2 2 2 2 3 2 2 3 2 2 3 3 1 2 1
 [1585] 2 2 2 3 2 2 2 1 2 2 2 2 3 2 2 3 3 2 3 2 2 2 3 2 2 1 2 2 3 2 2 2 3 2 2 2
 [1621] 3 2 3 3 2 3 2 2 2 3 3 2 2 3 2 2 2 2 3 2 2 3 2 3 2 2 2 2 2 2 3 3 2 3 2 2
 [1657] 2 1 2 2 2 2 2 1 1 3 2 3 2 1 2 1 2 3 2 2 2 3 2 3 2 2 1 3 2 2 2 3 3 2 3 2
 [1693] 2 3 2 2 2 2 2 2 2 2 2 2 2 3 1 2 2 3 2 2 2 2 2 3 2 2 3 2 2 2 2 2 3 2 2 2
 [1729] 2 3 2 1 3 2 2 2 3 1 3 1 3 2 2 2 3 1 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3
 [1765] 2 3 3 1 3 2 3 1 2 2 3 2 2 3 2 2 3 2 2 2 2 2 2 2 2 1 2 3 1 2 1 2 2 2 3 2
 [18

On vas fair tourner le model sur la full data puis le metre dans un fichier csv pour driven data

In [101]:
tic()
grid_default <- expand.grid(
  nrounds = 100,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_full <- caret::train(
    x = X,
    y = as.factor(y),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

76.656 sec elapsed


In [107]:
pred_base_full <- predict(xgb_base_full, true.test)

In [109]:
submission_format[2] <- as.numeric(pred_base_full)

In [111]:
write.csv(submission_format, file = "submission_format_xgboost_1.csv", row.names = FALSE)

tuning euritique avec un plus petit dataset

Pour 10000 data

In [ ]:
#size <- nrow(X)
#spliting_idx <- sample(1:size)
#split <- floor(size*0.8)

#X.train <- X[spliting_idx[1:split],]
#X.test <- X[spliting_idx[(split+1):size],]
#y.train <- y[spliting_idx[1:split]]
#y.test <- y[spliting_idx[(split+1):size]]

In [117]:
X.train.10000 <- X.train[1:10000,]
y.train.10000 <- y.train[1:10000]
X.test.1000 <- X.test[1:1000,]
y.test.1000 <- y.test[1:1000]

In [120]:
tic()
grid_default <- expand.grid(
  nrounds = c(200,400),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_10000 <- caret::train(
    x = X.train.10000,
    y = as.factor(y.train.10000),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE,
    nthread = 4
)
toc()

[17:15:41] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:15:50] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:16:00] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:16:09] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:16:19] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
53.564 sec elapsed


In [122]:
xgb_10000$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,200,6,0.3,0,1,1,1


In [121]:
pred_base <- predict(xgb_10000, X.test.1000)
ll <- length(y.test.1000)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test.1000[j]+1)] <- cfm[pred_base[j],(y.test.1000[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.702


In [123]:
tic()
grid_default <- expand.grid(
  nrounds = (1:4)*50,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_10000 <- caret::train(
    x = X.train.10000,
    y = as.factor(y.train.10000),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE,
    nthread = 4
)
toc()

[17:18:10] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:10] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:10] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:15] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:15] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:15] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:19] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:19] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:19] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:18:24] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [124]:
xgb_10000$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,50,6,0.3,0,1,1,1


In [125]:
pred_base <- predict(xgb_10000, X.test.1000)
ll <- length(y.test.1000)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test.1000[j]+1)] <- cfm[pred_base[j],(y.test.1000[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.715


Pour 50000 data

In [126]:
X.train.50000 <- X.train[1:50000,]
y.train.50000 <- y.train[1:50000]
X.test.5000 <- X.test[1:5000,]
y.test.5000 <- y.test[1:5000]

In [127]:
tic()
grid_default <- expand.grid(
  nrounds = (1:4)*50,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_50000 <- caret::train(
    x = X.train.50000,
    y = as.factor(y.train.50000),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE,
    nthread = 4
)
toc()

[17:21:36] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:21:36] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:21:36] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:21:59] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:21:59] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:21:59] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:22:22] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:22:22] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:22:22] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:22:45] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [128]:
xgb_50000$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2,100,6,0.3,0,1,1,1


In [130]:
pred_base <- predict(xgb_50000, X.test.5000)
ll <- length(y.test.5000)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test.5000[j]+1)] <- cfm[pred_base[j],(y.test.5000[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7394


In [142]:
tic()
grid_default <- expand.grid(
  nrounds = (2:12)*10,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_50000 <- caret::train(
    x = X.train.50000,
    y = as.factor(y.train.50000),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE,
    nthread = 4
)
toc()

[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:46:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [143]:
xgb_50000$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
6,70,6,0.3,0,1,1,1


In [144]:
pred_base <- predict(xgb_50000, X.test.5000)
ll <- length(y.test.5000)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test.5000[j]+1)] <- cfm[pred_base[j],(y.test.5000[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7406


pour 100000 data

In [132]:
X.train.100000 <- X.train[1:100000,]
y.train.100000 <- y.train[1:100000]
X.test.10000 <- X.test[1:10000,]
y.test.10000 <- y.test[1:10000]

In [135]:
tic()
grid_default <- expand.grid(
  nrounds = (7:13)*10,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_100000 <- caret::train(
    x = X.train.100000,
    y = as.factor(y.train.100000),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE,
    nthread = 4
)
toc()

[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:31:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:32:28] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:32:28] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:32:28] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:32:28] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [136]:
xgb_100000$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2,80,6,0.3,0,1,1,1


In [137]:
pred_base <- predict(xgb_100000, X.test.10000)
ll <- length(y.test.10000)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test.10000[j]+1)] <- cfm[pred_base[j],(y.test.10000[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7511


sur le full data set on teste avec 60,70,80

In [146]:
tic()
grid_default <- expand.grid(
  nrounds = c(60,70,80),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

[17:51:14] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:51:15] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:51:54] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:51:54] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:52:33] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:52:33] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:53:12] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:53:12] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:53:51] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:53:51] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [147]:
xgb_base$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3,80,6,0.3,0,1,1,1


In [148]:
pred_base <- predict(xgb_base, X.test)
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7514821


In [ ]:
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

In [149]:
tic()
grid_default <- expand.grid(
  nrounds = c(80,90,100),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_2 <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

[17:56:52] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:56:52] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:57:40] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:57:40] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:58:29] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:58:29] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:59:18] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[17:59:18] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:00:07] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:00:07] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [150]:
xgb_base_2$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3,100,6,0.3,0,1,1,1


In [151]:
pred_base <- predict(xgb_base_2, X.test)
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7520577


In [152]:
tic()
grid_default <- expand.grid(
  nrounds = c(100,110,120),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_3 <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

[18:04:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:04:03] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:05:01] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:05:01] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:06:00] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:06:01] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:06:59] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:06:59] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:07:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:07:57] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [153]:
xgb_base_3$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2,110,6,0.3,0,1,1,1


In [154]:
pred_base <- predict(xgb_base_3, X.test)
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7522688


In [156]:
tic()
grid_default <- expand.grid(
  nrounds = c(105,110,115),
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_4 <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

[18:12:56] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:12:56] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:13:51] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:13:52] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:14:47] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:14:47] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:15:43] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:15:43] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:16:39] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is deprecated, use `iteration_range` instead.
[18:16:39] WARNING: src/c_api/c_api.cc:935: `ntree_limit` is dep

In [157]:
xgb_base_4$bestTune

,nrounds,max_depth,eta,gamma,colsample_bytree,min_child_weight,subsample
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,105,6,0.3,0,1,1,1


In [158]:
pred_base <- predict(xgb_base_4, X.test)
ll <- length(y.test)
cfm <- matrix(0, nrow = 3, ncol = 3)
for (j in 1:ll){cfm[pred_base[j],(y.test[j]+1)] <- cfm[pred_base[j],(y.test[j]+1)]+1}
TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
P_micro <- sum(TP)/sum(TP+FP)
R_micro <- sum(TP)/sum(TP+FN)
print((2*P_micro*R_micro)/(P_micro+R_micro))

[1] 0.7523647


On va dire que nrounds = 100 c'est déjà bien comme ça !

Passon au reste des paramétre 

In [159]:
tic()
grid_default <- expand.grid(
  nrounds = 100,
  max_depth = 6,
  eta = c(0.025, 0.05, 0.1, 0.3),
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "cv",
    number = 5,
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_5 <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

#eta = c(0.025, 0.05, 0.1, 0.3),
#  max_depth = c(2, 3, 4, 5, 6),

Est ce qu'on enleve les geo_lvl avec le target encoding ?

In [5]:
library(xgboost)
library(Matrix)
library(ggplot2)
library(lattice)
library(caret)
library(dplyr)
library(tictoc)


Attachement du package : ‘dplyr’


L'objet suivant est masqué depuis ‘package:xgboost’:

    slice


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
data_target_encoding <- read.csv("data_target_encoding.csv",stringsAsFactors = T)
test_target_encoding <- read.csv("test_target_encoding.csv",stringsAsFactors = T)

In [33]:
data_target_encoding <- data_target_encoding[,-c()]

ERROR: Error in -c("building_id", "geo_level_1_id", "geo_level_2_id", "geo_level_3_id"): argument incorrect pour un opérateur unitaire


In [35]:
data_target_encoding <- data_target_encoding[,-c(1,2,5,8)]
test_target_encoding <- test_target_encoding[,-c(1,2,5,8)]

In [3]:
data_target_encoding[1:3,]

,geo_level_1_mean_damage,geo_level_1_sd_damage,geo_level_2_mean_damage,geo_level_2_sd_damage,geo_level_3_mean_damage,geo_level_3_sd_damage,count_floors_pre_eq,age,area_percentage,height_percentage,⋯,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,2.020477,0.4558226,1.984293,0.2987067,1.971429,0.1690309,1,25,5,2,⋯,0,0,0,0,0,0,0,0,0,2
2,2.794480,0.4352255,2.977444,0.1490457,3.000000,0.0000000,2,0,13,7,⋯,0,0,0,0,0,0,0,0,0,3
3,2.794480,0.4352255,2.984772,0.1227723,3.000000,0.0000000,2,5,12,6,⋯,0,0,0,0,0,0,0,0,0,3


In [24]:
test_target_encoding[1:3,]

,geo_level_1_id,geo_level_1_mean_damage,geo_level_1_sd_damage,geo_level_2_id,geo_level_2_mean_damage,geo_level_2_sd_damage,geo_level_3_id,geo_level_3_mean_damage,geo_level_3_sd_damage,count_floors_pre_eq,⋯,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,17,2.794480,0.4352255,596,2.705036,0.4732441,11307,2.631579,0.5972647,3,⋯,0,0,0,0,0,0,0,0,0,0
2,6,2.161724,0.5554311,141,2.180851,0.3859225,11987,2.000000,0.0000000,2,⋯,1,0,0,0,0,0,0,0,0,0
3,22,2.000960,0.5103307,19,2.175000,0.3848076,10044,3.000000,0.0000000,2,⋯,0,0,0,0,0,0,0,0,0,0


In [6]:
X.data_target_encoding <- sparse.model.matrix(damage_grade ~ .,data = data_target_encoding)[,-1]
#X.test_target_encoding <- sparse.model.matrix( ~ .,data = test_target_encoding)[,-1]
y.data_target_encoding <- as.numeric(data_target_encoding[,c("damage_grade")]-1)

In [27]:
X.data_target_encoding[1:3,]

  [[ suppressing 66 column names ‘geo_level_1_id’, ‘geo_level_1_mean_damage’, ‘geo_level_1_sd_damage’ ... ]]



3 x 66 sparse Matrix of class "dgCMatrix"
                                                                               
1 30 2.020477 0.4558226 266 1.984293 0.2987067  1224 1.971429 0.1690309 1 25  5
2 17 2.794480 0.4352255 409 2.977444 0.1490457 12182 3.000000 .         2  . 13
3 17 2.794480 0.4352255 716 2.984772 0.1227723  7056 3.000000 .         2  5 12
                                                                               
1 2 . 1 . 1 . . . . . . . . . . . . 1 . . 1 . . . . . . . . 1 . . . . . . . . .
2 7 . 1 . 1 . . . . . . . . 1 . . . 1 . . 1 . . . . . . . . 1 . . . . . . . . .
3 6 1 . . 1 . . 1 . . . . . 1 . . . 1 . . 1 . . . . . . . . 1 . . . . . . . . .
                               
1 . 1 . . . . . . . . . . . . .
2 . 1 . 1 . . . . . . . . . . .
3 . 1 . 1 . . . . . . . . . . .

In [28]:
y.data_target_encoding[1:3]

[1] 1 2 2

In [38]:
tic()
grid_default <- expand.grid(
  nrounds = 100,
  max_depth = 6,
  eta = 0.3,
  gamma = 0,
  colsample_bytree = 1,
  min_child_weight = 1,
  subsample = 1
)

train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
)

xgb_base_4 <- caret::train(
    x = X.data_target_encoding,
    y = as.factor(y.data_target_encoding),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
)
toc()

138.715 sec elapsed


In [39]:
pred_base_full <- predict(xgb_base_4, X.test_target_encoding)
submission_format[2] <- as.numeric(pred_base_full)
write.csv(submission_format, file = "submission_format_xgboost_target_encoding_2.csv", row.names = FALSE)

In [10]:
k = 10
accuracy_vec <- array(0,k)


# 1. Shuffle the dataset randomly.
spam_idx <- sample(1:nrow(X.data_target_encoding))

# 2. Split the dataset into k groups
max <- ceiling(nrow(X.data_target_encoding)/k)
splits <- split(spam_idx, ceiling(seq_along(spam_idx)/max)) # permet de gérer le fait qu'on split en des group de tail differente

pb2 <- txtProgressBar(min = 1, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){
    #3.1 Take the group as a hold out or test data set
    X.data_target_encoding.test <- X.data_target_encoding[splits[[i]],]
    y.data_target_encoding.test <- y.data_target_encoding[splits[[i]]]
    X.data_target_encoding.train <- X.data_target_encoding[-splits[[i]],]
    y.data_target_encoding.train <- y.data_target_encoding[-splits[[i]]]
    
    grid_default <- expand.grid(
    nrounds = 100,
    max_depth = 6,
    eta = 0.3,
    gamma = 0,
    colsample_bytree = 1,
    min_child_weight = 1,
    subsample = 1
    )

    train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
    )

    xgb_base_4 <- caret::train(
    x = X.data_target_encoding.train,
    y = as.factor(y.data_target_encoding.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    )

    pred_base <- predict(xgb_base_4, X.data_target_encoding.test)
    ll <- length(y.data_target_encoding.test)
    cfm <- matrix(0, nrow = 3, ncol = 3)
    for (j in 1:ll){cfm[pred_base[j],(y.data_target_encoding.test[j]+1)] <- cfm[pred_base[j],(y.data_target_encoding.test[j]+1)]+1}
    TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
    FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
    FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
    P_micro <- sum(TP)/sum(TP+FP)
    R_micro <- sum(TP)/sum(TP+FN)
    
    accuracy_vec[i] <- (2*P_micro*R_micro)/(P_micro+R_micro)
    
    setTxtProgressBar(pb2, i)
    print((2*P_micro*R_micro)/(P_micro+R_micro))   


}

  |                                                                      |   0%[1] 0.7644373
  |========                                                              |  11%[1] 0.761521
  |================                                                      |  22%[1] 0.7621734
  |=======================                                               |  33%[1] 0.7623268
  |===============================                                       |  44%[1] 0.7569932
  |=======================================                               |  56%[1] 0.762864
  |===============================================                       |  67%[1] 0.7600246
  |======================================================                |  78%[1] 0.7630175
  |==============================================================        |  89%[1] 0.7635164
  |======================================================================| 100%[1] 0.7591356


In [11]:
mean(accuracy_vec)

[1] 0.761601